# Import Packages

In [1]:
import os
import re
import logging
import numpy as np
from io import StringIO
from sklearn.metrics import confusion_matrix, classification_report
from IPython.display import clear_output

from bci_essentials.io.xdf_sources import XdfEegSource, XdfMarkerSource
from bci_essentials.bci_controller import BciController
from bci_essentials.paradigm.ssvep_paradigm import SsvepParadigm
from bci_essentials.data_tank.data_tank import DataTank
from bci_essentials.utils.logger import Logger  # Logger wrapper
from bci_essentials.classification.ssvep_fbcca_classifier import (
    SsvepFbCcaClassifier,
)

# Import Data & Initialize Objects

In [2]:
filename = os.path.join("data", "sub-P008_ses-S001_task-T2_run-001_eeg.xdf")

logger = Logger(name=__name__)
logger.setLevel(1)
eeg_source = XdfEegSource(filename)
marker_source = XdfMarkerSource(filename)
paradigm = SsvepParadigm(live_update=False, iterative_training=False) # Setting live_upadate to False means it will classify each trial rather than each epoch (full 5 seconds rather than every second 5 time)
data_tank = DataTank()

2025-06-16 09:54:49 - INFO - __main__ : Setting logging level to Unknown Level


# Define the classifier

In [3]:
# Settings
target_frequencies = np.array([6.25, 10, 11.11111, 14.28571])

n_harmonics = 3  # Number of harmonics to use for SSVEP classification
fb_cutoffs = np.array([[i, 31] for i in range(3, 29, 2)])   # Filter bank cut-off frequencies
#filter_order = 6
nsamples = 5 * 256  # 5 seconds of data at 256 Hz
fsample = 256.0  # Hz
concatenate_trials = True  # Concatenate trials for training

classifier = SsvepFbCcaClassifier(subset=["O1", "Oz", "O2"])
#classifier = SsvepFbCcaClassifier(subset=[])
classifier.set_ssvep_settings(
    fsample=fsample,
    n_harmonics=n_harmonics,  # Number of harmonics to use for SSVEP classification
    target_freqs=target_frequencies,
    n_samples=nsamples,
    filter_bank=fb_cutoffs,
    concatenate_trials=concatenate_trials,
    #filter_order=filter_order,
)

2025-06-16 09:54:50 - INFO - bci_essentials.classification.generic_classifier : Initializing the classifier


# Set up BCIController

In [4]:
test_ssvep = BciController(classifier, eeg_source, marker_source, paradigm, data_tank)

test_ssvep.setup(
    online=False,
    train_complete=True,  # Set to True to run the classifier on the entire dataset
)

2025-06-16 09:54:50 - INFO - bci_essentials.bci_controller : g.USBamp-1
2025-06-16 09:54:50 - INFO - bci_essentials.bci_controller : ['Fz', 'F4', 'F8', 'C3', 'Cz', 'C4', 'T8', 'P7', 'P3', 'P4', 'P8', 'PO7', 'PO8', 'O1', 'Oz', 'O2']


# Run

In [ ]:
# Create a string buffer to capture log output
log_stream = StringIO()
handler = logging.StreamHandler(log_stream)
logger = logging.getLogger()
logger.addHandler(handler)

# Run the test
test_ssvep.run()

# Get log output as string
log_output = log_stream.getvalue()

# Extract true and predicted labels using regex
y_true = []
y_pred = []

for line in log_output.split('\n'):
    # Extract true labels
    if "Cued index: " in line:
        if "Processed" in line: #do not save the "WARNING" lines
            true_label = int(re.search(r"Cued index: (\d+)", line).group(1))
            y_true.append(true_label)
    
    # Extract predicted labels
    if "Predicted label: " in line:
        pred_label = int(re.search(r"Predicted label: (\d+)", line).group(1))
        y_pred.append(pred_label)

clear_output()

# Convert lists to numpy arrays
#y_true = np.array(y_true[:-1])
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate and display accuracy
accuracy = np.mean(y_true == y_pred)
print(f"Accuracy: {accuracy:.2%}")

# Create confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(conf_matrix)

# Print detailed classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred))
print(f"Number of samples: {len(y_true)}")
print(f"True labels: {y_true}")
print(f"Predicted labels: {y_pred}")

Accuracy: 100.00%

Confusion Matrix:
[[8 0 0 0]
 [0 9 0 0]
 [0 0 8 0]
 [0 0 0 5]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         8
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00         8
           3       1.00      1.00      1.00         5

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

Number of samples: 30
True labels: [1 0 2 3 0 0 2 1 0 2 2 1 3 1 2 1 3 3 1 3 2 2 1 0 0 1 0 1 2 0]
Predicted labels: [1 0 2 3 0 0 2 1 0 2 2 1 3 1 2 1 3 3 1 3 2 2 1 0 0 1 0 1 2 0]
